
# NB_05 — Payments Incremental Bronze-to-Silver Processing

## Purpose

This notebook implements a **production-style incremental ETL pipeline** for insurance **Payments** data, moving records from the **Bronze Lakehouse** into a validated and standardized **Silver Delta table**.

It extends the reusable, metadata-driven incremental processing framework already implemented for **Customers, Policies, and Claims**.

---

## Architecture

**Source:** `LH_Bronze.dbo.bronze_payments`  
**Target:** `LH_Silver.dbo.silver_payments`  
**Control Table:** `LH_Silver.dbo.etl_control`  
**Audit Table:** `LH_Silver.dbo.etl_batch_audit`  
**Business Key:** `payment_id`  
**Watermark:** `last_updated`  
**Load Type:** `INCREMENTAL`

---

## Processing Flow

`Bronze Payments`

↓  
**Read ETL Control Metadata**

↓  
**Retrieve Stored Watermark**

↓  
**Extract New / Changed Payments**

↓  
**Data Quality Validation**

↓  
**Reject Invalid Records**

↓  
**Deduplicate by `payment_id`**

↓  
**Standardize and Transform to Silver Schema**

↓  
**Compare with Existing Silver Payments**

↓  
**Classify Changes**

`INSERT` | `UPDATE` | `NO-OP`

↓  
**Delta Lake MERGE**

↓  
**Write Batch Audit**

↓  
**Advance Watermark After Successful Processing**

↓  
**Restart / Idempotency Verification**

---

## Production Design Principles

This notebook demonstrates several production data-engineering patterns:

- **Metadata-driven processing** — source, target, watermark, and load behavior are controlled through ETL metadata.
- **Incremental ingestion** — only records newer than the persisted watermark are processed.
- **Data quality controls** — invalid Payments are rejected before reaching Silver.
- **Business-key deduplication** — multiple versions of a Payment are resolved using `payment_id`.
- **Change detection** — incoming records are classified as `INSERT`, `UPDATE`, or `NO-OP`.
- **Delta Lake upsert** — genuine changes are persisted using an idempotent `MERGE`.
- **Auditability** — every execution records source, insert, update, reject, and execution status.
- **Safe watermark advancement** — the watermark advances only after successful Silver processing.
- **Restart safety** — rerunning the pipeline does not create duplicate records or repeat previously processed changes.

---

## Expected Outcome

At completion, `NB_05_Payments_Incremental` will provide a **restart-safe, auditable, incremental Bronze-to-Silver Payments pipeline** using Microsoft Fabric, PySpark, Delta Lake, metadata-driven control, data-quality validation, and operational auditing.


In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

# ============================================================
# STEP 1 - INITIALIZE PAYMENTS INCREMENTAL PIPELINE
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

from datetime import datetime
import uuid

PIPELINE_NAME = "PL_Insurance_Medallion_ETL"
SOURCE_NAME = "PAYMENTS"

BATCH_ID = str(uuid.uuid4())
RUN_START_TS = datetime.now()

SOURCE_TABLE = "LH_Bronze.dbo.bronze_payments"
TARGET_TABLE = "LH_Silver.dbo.silver_payments"

CONTROL_TABLE = "LH_Silver.dbo.etl_control"
AUDIT_TABLE = "LH_Silver.dbo.etl_batch_audit"

BUSINESS_KEY = "payment_id"
WATERMARK_COLUMN = "last_updated"

print("Payments incremental pipeline initialized.")
print("----------------------------------------")
print(f"Pipeline       : {PIPELINE_NAME}")
print(f"Source         : {SOURCE_NAME}")
print(f"Batch ID       : {BATCH_ID}")
print(f"Run start      : {RUN_START_TS}")
print(f"Bronze table   : {SOURCE_TABLE}")
print(f"Silver table   : {TARGET_TABLE}")
print(f"Business key   : {BUSINESS_KEY}")
print(f"Watermark      : {WATERMARK_COLUMN}")

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 3, Finished, Available, Finished, False)

Payments incremental pipeline initialized.
----------------------------------------
Pipeline       : PL_Insurance_Medallion_ETL
Source         : PAYMENTS
Batch ID       : e1ed9e2a-52d3-48cd-bff8-11cbdf960abe
Run start      : 2026-08-22 13:45:00.014596
Bronze table   : LH_Bronze.dbo.bronze_payments
Silver table   : LH_Silver.dbo.silver_payments
Business key   : payment_id
Watermark      : last_updated


## Step 2 — Inspect Bronze Payments Source

Before implementing incremental processing, inspect the physical Bronze Payments
table and validate its schema.

This step verifies:

- The Bronze Payments table is accessible.
- The expected business key `payment_id` exists.
- The incremental watermark column `last_updated` exists.
- The available source columns and their current data types are understood.
- The Bronze row count establishes the baseline for subsequent reconciliation.

No data is modified in this step.

In [3]:

# ============================================================
# STEP 2 - INSPECT ACTUAL BRONZE PAYMENTS SCHEMA
# ============================================================

bronze_payments_df = spark.table(SOURCE_TABLE)

bronze_payments_count = bronze_payments_df.count()

print("Bronze Payments source inspected.")
print("----------------------------------------")
print(f"Table       : {SOURCE_TABLE}")
print(f"Bronze rows : {bronze_payments_count}")

print()
print("Bronze Payments columns:")
print("----------------------------------------")

for c in bronze_payments_df.columns:
    print(c)

print()
print("Bronze Payments schema:")
print("----------------------------------------")

bronze_payments_df.printSchema()

print()
print("Sample Payments records:")
print("----------------------------------------")

display(
    bronze_payments_df.limit(10)
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 5, Finished, Available, Finished, False)

Bronze Payments source inspected.
----------------------------------------
Table       : LH_Bronze.dbo.bronze_payments
Bronze rows : 1501

Bronze Payments columns:
----------------------------------------
payment_id
policy_id
customer_id
claim_id
payment_date
payment_type
payment_amount
payment_method
payment_status
transaction_reference

Bronze Payments schema:
----------------------------------------
root
 |-- payment_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- claim_id: string (nullable = true)
 |-- payment_date: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- transaction_reference: string (nullable = true)


Sample Payments records:
----------------------------------------


SynapseWidget(Synapse.DataFrame, 20433352-9398-43af-a0c4-c13867e10a3d)

## Step 2A — Establish Payments Incremental Watermark

The source Payments dataset does not contain a technical modification timestamp.

`payment_date` represents the business date of the payment and is therefore
not suitable as a reliable incremental-processing watermark. A payment created
on an earlier date may subsequently change status or other business attributes.

To support reliable incremental processing, this implementation introduces
`last_updated` as the technical change-tracking column.

For the existing historical Bronze dataset, `payment_date` is used to initialize
`last_updated`. Future controlled INSERT and UPDATE test records will explicitly
set `last_updated` to represent their actual modification timestamp.

This enables the pipeline to detect both:

- Newly created Payments
- Changes to previously existing Payments

without relying solely on the original business transaction date.

In [4]:
# ============================================================
# STEP 2A - ADD TECHNICAL WATERMARK TO BRONZE PAYMENTS
# ============================================================

from pyspark.sql import functions as F

payments_before_df = spark.table(SOURCE_TABLE)
payments_before_count = payments_before_df.count()

print("Preparing Payments technical watermark.")
print("----------------------------------------")
print(f"Bronze rows              : {payments_before_count}")
print(f"Existing last_updated    : {'last_updated' in payments_before_df.columns}")


# ------------------------------------------------------------
# Add last_updated only if it does not already exist
# ------------------------------------------------------------

if "last_updated" not in payments_before_df.columns:

    payments_with_watermark_df = (
        payments_before_df
        .withColumn(
            "last_updated",
            F.to_date(F.col("payment_date"))
        )
    )

    (
        payments_with_watermark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SOURCE_TABLE)
    )

    print()
    print("last_updated added to Bronze Payments.")

else:
    print()
    print("last_updated already exists. No schema change required.")


# ------------------------------------------------------------
# Reload and verify persisted Bronze table
# ------------------------------------------------------------

bronze_payments_df = spark.table(SOURCE_TABLE)

payments_after_count = bronze_payments_df.count()

assert "last_updated" in bronze_payments_df.columns, \
    "Payments last_updated column was not created."

assert payments_after_count == payments_before_count, \
    "Bronze Payments row count changed during schema enhancement."

null_watermarks = (
    bronze_payments_df
    .filter(F.col("last_updated").isNull())
    .count()
)

assert null_watermarks == 0, \
    f"Found {null_watermarks} Payments with NULL last_updated."


print()
print("==========================================")
print(" PAYMENTS WATERMARK INITIALIZATION PASSED")
print("==========================================")
print(f"Rows before       : {payments_before_count}")
print(f"Rows after        : {payments_after_count}")
print(f"NULL watermarks   : {null_watermarks}")

print()
bronze_payments_df.printSchema()

display(
    bronze_payments_df
    .select(
        "payment_id",
        "payment_date",
        "payment_status",
        "payment_amount",
        "last_updated"
    )
    .orderBy(F.col("last_updated").desc())
    .limit(10)
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 6, Finished, Available, Finished, False)

Preparing Payments technical watermark.
----------------------------------------
Bronze rows              : 1501
Existing last_updated    : False

last_updated added to Bronze Payments.

 PAYMENTS WATERMARK INITIALIZATION PASSED
Rows before       : 1501
Rows after        : 1501
NULL watermarks   : 0

root
 |-- payment_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- claim_id: string (nullable = true)
 |-- payment_date: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- transaction_reference: string (nullable = true)
 |-- last_updated: date (nullable = true)



SynapseWidget(Synapse.DataFrame, 21088f0b-0781-41e5-bc51-18318ddba7eb)


## Step 3 — Register Payments ETL Control Metadata

The Payments source now contains the technical `last_updated` watermark required
for incremental processing.

This step registers `PAYMENTS` in the shared ETL control framework.

The control metadata defines:

- Bronze source table
- Silver target table
- Watermark column
- Initial persisted watermark
- Incremental load type
- Active processing status

For the initial Payments load, the watermark begins at `1900-01-01`.
This allows all existing Bronze Payments to participate in the first controlled
incremental execution.

After a successful Silver load, the watermark will advance to the maximum
successfully processed `last_updated` value.

This metadata-driven design allows the same incremental processing framework
to be reused across Customers, Policies, Claims, and Payments.

In [6]:
# ============================================================
# STEP 3 - REGISTER / LOAD PAYMENTS ETL CONTROL METADATA
# ============================================================

from pyspark.sql import functions as F

INITIAL_PAYMENT_WATERMARK = "1900-01-01 00:00:00"

# ------------------------------------------------------------
# 1. Check whether PAYMENTS already exists
# ------------------------------------------------------------

payments_control_existing_df = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("source_name") == SOURCE_NAME)
)

payments_control_existing_count = payments_control_existing_df.count()

print("Inspecting Payments ETL control metadata.")
print("-----------------------------------------")
print(f"Existing PAYMENTS control rows : {payments_control_existing_count}")


# ------------------------------------------------------------
# 2. Insert configuration only when missing
# ------------------------------------------------------------

if payments_control_existing_count == 0:

    spark.sql(f"""
        INSERT INTO {CONTROL_TABLE}
        (
            source_name,
            source_table,
            target_table,
            watermark_column,
            load_type,
            last_watermark,
            is_active,
            _created_ts,
            _updated_ts
        )
        VALUES
        (
            '{SOURCE_NAME}',
            '{SOURCE_TABLE}',
            '{TARGET_TABLE}',
            '{WATERMARK_COLUMN}',
            'INCREMENTAL',
            TIMESTAMP('{INITIAL_PAYMENT_WATERMARK}'),
            true,
            current_timestamp(),
            current_timestamp()
        )
    """)

    print()
    print("PAYMENTS control metadata created.")

else:

    print()
    print("PAYMENTS control metadata already exists.")
    print("Existing metadata preserved.")


# ------------------------------------------------------------
# 3. Reload active configuration
# ------------------------------------------------------------

payments_config_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
)

payments_config_count = payments_config_df.count()

assert payments_config_count == 1, \
    f"Expected exactly one active PAYMENTS configuration, found {payments_config_count}."


payments_config = payments_config_df.first()


# ------------------------------------------------------------
# 4. Read metadata dynamically
# ------------------------------------------------------------

CONFIG_SOURCE_TABLE = payments_config["source_table"]
CONFIG_TARGET_TABLE = payments_config["target_table"]
CONFIG_WATERMARK_COLUMN = payments_config["watermark_column"]
LAST_WATERMARK = payments_config["last_watermark"]
LOAD_TYPE = payments_config["load_type"]


# ------------------------------------------------------------
# 5. Verify configuration
# ------------------------------------------------------------

assert CONFIG_SOURCE_TABLE == SOURCE_TABLE
assert CONFIG_TARGET_TABLE == TARGET_TABLE
assert CONFIG_WATERMARK_COLUMN == WATERMARK_COLUMN
assert LOAD_TYPE == "INCREMENTAL"


print()
print("==========================================")
print(" PAYMENTS ETL CONFIGURATION LOADED")
print("==========================================")
print(f"Source name       : {SOURCE_NAME}")
print(f"Source table      : {CONFIG_SOURCE_TABLE}")
print(f"Target table      : {CONFIG_TARGET_TABLE}")
print(f"Watermark column  : {CONFIG_WATERMARK_COLUMN}")
print(f"Stored watermark  : {LAST_WATERMARK}")
print(f"Load type         : {LOAD_TYPE}")

display(payments_config_df)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 8, Finished, Available, Finished, False)

Inspecting Payments ETL control metadata.
-----------------------------------------
Existing PAYMENTS control rows : 1

PAYMENTS control metadata already exists.
Existing metadata preserved.


AssertionError: 

### Step 3A — Inspect Existing Payments Control Metadata

Before modifying the ETL control configuration, inspect the existing `PAYMENTS`
metadata to verify the currently registered source, target, watermark column,
load type, and stored watermark.

This validation identified that the existing configuration uses `payment_date`
as the watermark.

Because `payment_date` represents the business transaction date rather than
the technical modification timestamp, it is not sufficient for detecting
subsequent changes to existing payment records.

**Current configuration**

- Source: `LH_Bronze.dbo.bronze_payments`
- Target: `LH_Silver.dbo.silver_payments`
- Existing watermark column: `payment_date`
- Load type: `INCREMENTAL`
- Initial watermark: `1900-01-01`

The next step corrects the watermark configuration to use the newly established
technical `last_updated` column.

In [7]:

# ============================================================
# STEP 3A - INSPECT EXISTING PAYMENTS CONTROL METADATA
# ============================================================

payments_control_debug_df = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("source_name") == SOURCE_NAME)
)

print("Existing PAYMENTS control metadata:")
print("----------------------------------------")

display(payments_control_debug_df)

for row in payments_control_debug_df.collect():
    print(f"source_name      : {row['source_name']}")
    print(f"source_table     : {row['source_table']}")
    print(f"target_table     : {row['target_table']}")
    print(f"watermark_column : {row['watermark_column']}")
    print(f"load_type        : {row['load_type']}")
    print(f"last_watermark   : {row['last_watermark']}")
    print(f"is_active        : {row['is_active']}")
    print("----------------------------------------")

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 9, Finished, Available, Finished, False)

Existing PAYMENTS control metadata:
----------------------------------------


SynapseWidget(Synapse.DataFrame, 0976b63b-016b-4dcc-8fe1-142f9013b711)

source_name      : PAYMENTS
source_table     : LH_Bronze.dbo.bronze_payments
target_table     : LH_Silver.dbo.silver_payments
watermark_column : payment_date
load_type        : INCREMENTAL
last_watermark   : 1900-01-01 00:00:00
is_active        : True
----------------------------------------


## Step 3B — Correct Payments Watermark Metadata

The existing Payments control configuration was originally registered using
`payment_date` as the incremental watermark.

Source inspection showed that `payment_date` represents the business transaction
date rather than the technical modification timestamp.

The Bronze Payments source has now been enhanced with `last_updated` to support
reliable detection of both newly created and subsequently modified Payments.

Because the Payments pipeline has not yet processed an incremental batch and its
stored watermark remains at the initial `1900-01-01`, the control metadata can
safely be corrected before processing begins.

This step changes only the metadata configuration:

`payment_date` → `last_updated`

The stored watermark remains unchanged at `1900-01-01`.

In [9]:
# ============================================================
# STEP 3B - CORRECT PAYMENTS WATERMARK CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# 1. Update existing PAYMENTS metadata
# ------------------------------------------------------------

spark.sql(f"""
    UPDATE {CONTROL_TABLE}
       SET watermark_column = 'last_updated',
           _updated_ts = current_timestamp()
     WHERE source_name = '{SOURCE_NAME}'
       AND is_active = true
""")

print("PAYMENTS watermark metadata updated.")


# ------------------------------------------------------------
# 2. Reload configuration
# ------------------------------------------------------------

payments_config_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
)

payments_config_count = payments_config_df.count()

assert payments_config_count == 1, \
    f"Expected exactly one active PAYMENTS configuration, found {payments_config_count}."

payments_config = payments_config_df.first()


# ------------------------------------------------------------
# 3. Reload runtime configuration variables
# ------------------------------------------------------------

CONFIG_SOURCE_TABLE = payments_config["source_table"]
CONFIG_TARGET_TABLE = payments_config["target_table"]
CONFIG_WATERMARK_COLUMN = payments_config["watermark_column"]
LAST_WATERMARK = payments_config["last_watermark"]
LOAD_TYPE = payments_config["load_type"]


# ------------------------------------------------------------
# 4. Validate corrected configuration
# ------------------------------------------------------------

assert CONFIG_SOURCE_TABLE == SOURCE_TABLE, \
    "PAYMENTS source table configuration mismatch."

assert CONFIG_TARGET_TABLE == TARGET_TABLE, \
    "PAYMENTS target table configuration mismatch."

assert CONFIG_WATERMARK_COLUMN == "last_updated", \
    "PAYMENTS watermark configuration was not corrected."

assert LOAD_TYPE == "INCREMENTAL", \
    "PAYMENTS load type must be INCREMENTAL."

assert str(LAST_WATERMARK).startswith("1900-01-01"), \
    f"Unexpected PAYMENTS starting watermark: {LAST_WATERMARK}"


print()
print("==========================================")
print(" PAYMENTS ETL CONFIGURATION VERIFIED")
print("==========================================")
print(f"Source name       : {SOURCE_NAME}")
print(f"Source table      : {CONFIG_SOURCE_TABLE}")
print(f"Target table      : {CONFIG_TARGET_TABLE}")
print(f"Watermark column  : {CONFIG_WATERMARK_COLUMN}")
print(f"Stored watermark  : {LAST_WATERMARK}")
print(f"Load type         : {LOAD_TYPE}")

display(payments_config_df)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 11, Finished, Available, Finished, False)

PAYMENTS watermark metadata updated.

 PAYMENTS ETL CONFIGURATION VERIFIED
Source name       : PAYMENTS
Source table      : LH_Bronze.dbo.bronze_payments
Target table      : LH_Silver.dbo.silver_payments
Watermark column  : last_updated
Stored watermark  : 1900-01-01 00:00:00
Load type         : INCREMENTAL


SynapseWidget(Synapse.DataFrame, 5b423dc0-305e-4081-8c21-5b278a70c727)


## Step 4 — Extract Incremental Payments

Read the persisted Payments watermark from the ETL control framework and select
only Bronze records whose `last_updated` value is greater than the stored watermark.

For the initial execution, the stored watermark is `1900-01-01`, so all existing
Bronze Payments are expected to qualify for processing.

This step:

- Reads Payments from the Bronze Lakehouse.
- Applies the persisted `last_updated` watermark.
- Converts the watermark to a timestamp for consistent comparison.
- Identifies the incremental processing set.
- Performs reconciliation between the Bronze source and incremental selection.

No Silver data is modified in this step.

In [10]:
# ============================================================
# STEP 4 - EXTRACT INCREMENTAL PAYMENTS
# ============================================================

from pyspark.sql import functions as F

# Reload Bronze source
bronze_payments_df = spark.table(CONFIG_SOURCE_TABLE)

bronze_payments_count = bronze_payments_df.count()


# ------------------------------------------------------------
# Normalize technical watermark
# ------------------------------------------------------------

payments_with_watermark_df = (
    bronze_payments_df
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(CONFIG_WATERMARK_COLUMN))
    )
)


# ------------------------------------------------------------
# Validate watermark conversion
# ------------------------------------------------------------

invalid_watermark_count = (
    payments_with_watermark_df
    .filter(F.col("_watermark_ts").isNull())
    .count()
)

assert invalid_watermark_count == 0, \
    f"Found {invalid_watermark_count} invalid Payments watermarks."


# ------------------------------------------------------------
# Incremental extraction
# ------------------------------------------------------------

incremental_payments_df = (
    payments_with_watermark_df
    .filter(
        F.col("_watermark_ts") >
        F.lit(LAST_WATERMARK).cast("timestamp")
    )
)

incremental_payments_count = incremental_payments_df.count()


# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

assert incremental_payments_count <= bronze_payments_count, \
    "Incremental Payments count cannot exceed Bronze count."


print("Payments incremental extraction completed.")
print("------------------------------------------")
print(f"Bronze rows          : {bronze_payments_count}")
print(f"Stored watermark     : {LAST_WATERMARK}")
print(f"Incremental records  : {incremental_payments_count}")
print(f"Invalid watermarks   : {invalid_watermark_count}")


# ------------------------------------------------------------
# Preview newest incremental records
# ------------------------------------------------------------

display(
    incremental_payments_df
    .orderBy(F.col("_watermark_ts").desc())
    .limit(10)
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 12, Finished, Available, Finished, False)

Payments incremental extraction completed.
------------------------------------------
Bronze rows          : 1501
Stored watermark     : 1900-01-01 00:00:00
Incremental records  : 1501
Invalid watermarks   : 0


SynapseWidget(Synapse.DataFrame, 937ff794-e91f-4b00-b862-fe65059625b0)


## Step 5 — Validate Payments Data Quality

Validate the incremental Payments dataset before records are allowed to enter
the Silver processing layer.

The validation rules distinguish between structural requirements and
business-context requirements.

### Core validation rules

A valid Payment must contain:

- `payment_id`
- `policy_id`
- `customer_id`
- Valid `payment_date`
- Numeric `payment_amount`
- `payment_amount >= 0`
- `payment_type`
- `payment_method`
- `payment_status`
- `transaction_reference`
- Valid technical `last_updated`

### Conditional Claim validation

`claim_id` is not universally required.

- **Claim Payout** transactions require a `claim_id`.
- **Premium** transactions may legitimately have a NULL `claim_id`.

Records failing these rules are classified as rejects and excluded from the
Silver-ready dataset.

The reconciliation rule is:

**Incremental Records = Valid Records + Rejected Records**

No Silver data is modified during this step.

In [11]:

# ============================================================
# STEP 5 - PAYMENTS DATA QUALITY VALIDATION
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Parse fields required for validation
# ------------------------------------------------------------

payments_validation_df = (
    incremental_payments_df

    .withColumn(
        "_payment_date_parsed",
        F.to_date(F.col("payment_date"))
    )

    .withColumn(
        "_payment_amount_parsed",
        F.col("payment_amount").cast("double")
    )
)


# ------------------------------------------------------------
# Define validation rules
# ------------------------------------------------------------

invalid_condition = (

    # Required identifiers
    F.col("payment_id").isNull() |
    F.col("policy_id").isNull() |
    F.col("customer_id").isNull() |

    # Business date
    F.col("_payment_date_parsed").isNull() |

    # Amount
    F.col("_payment_amount_parsed").isNull() |
    (F.col("_payment_amount_parsed") < 0) |

    # Required business attributes
    F.col("payment_type").isNull() |
    F.col("payment_method").isNull() |
    F.col("payment_status").isNull() |
    F.col("transaction_reference").isNull() |

    # Technical watermark
    F.col("_watermark_ts").isNull() |

    # Claim Payout must reference a Claim
    (
        (F.upper(F.trim(F.col("payment_type"))) == "CLAIM PAYOUT") &
        F.col("claim_id").isNull()
    )
)


# ------------------------------------------------------------
# Split valid and rejected Payments
# ------------------------------------------------------------

valid_payments_df = (
    payments_validation_df
    .filter(~invalid_condition)
)

rejected_payments_df = (
    payments_validation_df
    .filter(invalid_condition)
)


valid_payments_count = valid_payments_df.count()
rejected_payments_count = rejected_payments_df.count()


# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

assert (
    valid_payments_count + rejected_payments_count
    == incremental_payments_count
), "Payments validation reconciliation failed."


print("Payments validation completed.")
print("------------------------------------------")
print(f"Incremental records : {incremental_payments_count}")
print(f"Valid records       : {valid_payments_count}")
print(f"Rejected records    : {rejected_payments_count}")
print(
    f"Reconciliation      : "
    f"{valid_payments_count + rejected_payments_count}"
)


# ------------------------------------------------------------
# Preview rejected Payments
# ------------------------------------------------------------

if rejected_payments_count > 0:

    print()
    print("Rejected Payments:")
    print("------------------------------------------")

    display(
        rejected_payments_df.select(
            "payment_id",
            "policy_id",
            "customer_id",
            "claim_id",
            "payment_date",
            "payment_type",
            "payment_amount",
            "payment_method",
            "payment_status",
            "transaction_reference",
            "last_updated"
        )
    )

else:
    print()
    print("No rejected Payments found.")

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 13, Finished, Available, Finished, False)

Payments validation completed.
------------------------------------------
Incremental records : 1501
Valid records       : 1500
Rejected records    : 1
Reconciliation      : 1501

Rejected Payments:
------------------------------------------


SynapseWidget(Synapse.DataFrame, 700a5219-a3c1-4638-b835-915610bc6f05)


## Step 6 — Deduplicate Valid Payments

Deduplicate the validated Payments dataset using `payment_id` as the business key.

Incremental source data may contain multiple versions of the same Payment due to
source-system corrections, retries, replayed files, or repeated ingestion.

For each `payment_id`, the record with the most recent `last_updated` value is retained.

This step:

- Processes only records that passed data-quality validation.
- Partitions records by `payment_id`.
- Orders each Payment version by `last_updated` descending.
- Retains the latest version of each Payment.
- Measures duplicate versions removed.
- Reconciles the deduplicated dataset against the valid input.

No Silver data is modified in this step.

In [12]:
# ============================================================
# STEP 6 - DEDUPLICATE VALID PAYMENTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ------------------------------------------------------------
# Define latest-version window
# ------------------------------------------------------------

payment_dedup_window = (
    Window
    .partitionBy("payment_id")
    .orderBy(
        F.col("_watermark_ts").desc(),
        F.col("transaction_reference").desc()
    )
)


# ------------------------------------------------------------
# Rank Payment versions
# ------------------------------------------------------------

ranked_payments_df = (
    valid_payments_df
    .withColumn(
        "_payment_version_rank",
        F.row_number().over(payment_dedup_window)
    )
)


# ------------------------------------------------------------
# Retain latest version
# ------------------------------------------------------------

unique_payments_df = (
    ranked_payments_df
    .filter(F.col("_payment_version_rank") == 1)
    .drop("_payment_version_rank")
)


unique_payments_count = unique_payments_df.count()

duplicate_versions_count = (
    valid_payments_count - unique_payments_count
)


# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

assert (
    unique_payments_count + duplicate_versions_count
    == valid_payments_count
), "Payments deduplication reconciliation failed."


assert (
    unique_payments_df
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
    == 0
), "Duplicate payment_id values remain after deduplication."


print("Payments deduplication completed.")
print("------------------------------------------")
print(f"Valid records       : {valid_payments_count}")
print(f"Unique Payments     : {unique_payments_count}")
print(f"Duplicate versions  : {duplicate_versions_count}")
print(
    f"Reconciliation      : "
    f"{unique_payments_count + duplicate_versions_count}"
)


# ------------------------------------------------------------
# Show duplicate Payment IDs if any existed
# ------------------------------------------------------------

duplicate_payment_ids_df = (
    ranked_payments_df
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)

display(duplicate_payment_ids_df)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 14, Finished, Available, Finished, False)

Payments deduplication completed.
------------------------------------------
Valid records       : 1500
Unique Payments     : 1499
Duplicate versions  : 1
Reconciliation      : 1500


SynapseWidget(Synapse.DataFrame, 41db0de0-5ba1-40d9-a623-65352aa5eb85)


## Step 7 — Transform Payments to Silver Schema

Transform the validated and deduplicated Payments dataset into the standardized
Silver-layer schema.

The Silver transformation applies consistent business and technical data types
and normalizes categorical attributes before persistence.

### Transformations

- Convert `payment_date` to `date`.
- Convert `payment_amount` to `double`.
- Convert `last_updated` to `timestamp`.
- Standardize `payment_type` to uppercase.
- Standardize `payment_method` to uppercase.
- Standardize `payment_status` to uppercase.
- Trim identifier and reference fields.
- Add `_silver_processed_ts` as the Silver processing timestamp.

Only the 1,499 validated and deduplicated Payments continue to this stage.

The rejected source record and superseded duplicate version remain excluded
from the Silver-ready dataset.

No Silver data is modified in this step.

In [13]:
# ============================================================
# STEP 7 - TRANSFORM PAYMENTS TO SILVER SCHEMA
# ============================================================

from pyspark.sql import functions as F


silver_ready_payments_df = (
    unique_payments_df

    .select(
        F.trim(F.col("payment_id")).alias("payment_id"),
        F.trim(F.col("policy_id")).alias("policy_id"),
        F.trim(F.col("customer_id")).alias("customer_id"),
        F.trim(F.col("claim_id")).alias("claim_id"),

        F.to_date(F.col("payment_date")).alias("payment_date"),

        F.upper(
            F.trim(F.col("payment_type"))
        ).alias("payment_type"),

        F.col("payment_amount")
         .cast("double")
         .alias("payment_amount"),

        F.upper(
            F.trim(F.col("payment_method"))
        ).alias("payment_method"),

        F.upper(
            F.trim(F.col("payment_status"))
        ).alias("payment_status"),

        F.trim(
            F.col("transaction_reference")
        ).alias("transaction_reference"),

        F.to_timestamp(
            F.col("last_updated")
        ).alias("last_updated"),

        F.current_timestamp().alias("_silver_processed_ts")
    )
)


# ------------------------------------------------------------
# Validate Silver-ready dataset
# ------------------------------------------------------------

silver_ready_payments_count = (
    silver_ready_payments_df.count()
)

assert (
    silver_ready_payments_count
    == unique_payments_count
), "Payments Silver transformation changed the row count."


# Ensure business key remains unique
duplicate_silver_keys = (
    silver_ready_payments_df
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_silver_keys == 0, \
    "Duplicate payment_id values detected in Silver-ready Payments."


print("Payments Silver transformation completed.")
print("------------------------------------------")
print(f"Unique valid Payments : {unique_payments_count}")
print(f"Silver-ready Payments : {silver_ready_payments_count}")

print()
print("Silver-ready Payments schema:")
silver_ready_payments_df.printSchema()


display(
    silver_ready_payments_df
    .orderBy(F.col("last_updated").desc())
    .limit(10)
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 15, Finished, Available, Finished, False)

Payments Silver transformation completed.
------------------------------------------
Unique valid Payments : 1499
Silver-ready Payments : 1499

Silver-ready Payments schema:
root
 |-- payment_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- claim_id: string (nullable = true)
 |-- payment_date: date (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- transaction_reference: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = false)



SynapseWidget(Synapse.DataFrame, 30b283ea-5863-4abf-a9df-7b0a6d50c114)

Your flow is now:





## Step 8 — Inspect Existing Silver Payments Target

Before performing the Delta MERGE, inspect the existing Silver Payments table
and compare its structure with the newly prepared Silver-ready dataset.

This pre-MERGE validation verifies:

- Whether the Silver target table already exists.
- Current Silver row count.
- Existing target schema and data types.
- Compatibility between source and target columns.
- Current uniqueness of the `payment_id` business key.

The target is inspected before any write operation to prevent schema conflicts
or unintended modifications.

No Silver data is modified in this step.

In [14]:
# ============================================================
# STEP 8 - INSPECT EXISTING SILVER PAYMENTS TARGET
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Load existing Silver target
# ------------------------------------------------------------

existing_silver_payments_df = spark.table(CONFIG_TARGET_TABLE)

existing_silver_payments_count = (
    existing_silver_payments_df.count()
)


print("Existing Silver Payments table inspected.")
print("------------------------------------------")
print(f"Target table  : {CONFIG_TARGET_TABLE}")
print(f"Existing rows : {existing_silver_payments_count}")

print()
print("Existing Silver Payments schema:")
print("------------------------------------------")

existing_silver_payments_df.printSchema()


# ------------------------------------------------------------
# Inspect existing duplicate business keys
# ------------------------------------------------------------

existing_silver_duplicate_keys_df = (
    existing_silver_payments_df
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
)

existing_silver_duplicate_key_count = (
    existing_silver_duplicate_keys_df.count()
)

print()
print(f"Duplicate payment_id keys : {existing_silver_duplicate_key_count}")


# ------------------------------------------------------------
# Compare source/target columns
# ------------------------------------------------------------

source_columns = set(silver_ready_payments_df.columns)
target_columns = set(existing_silver_payments_df.columns)

missing_in_target = sorted(source_columns - target_columns)
extra_in_target = sorted(target_columns - source_columns)


print()
print("Schema column comparison:")
print("------------------------------------------")
print(f"Missing in target : {missing_in_target}")
print(f"Extra in target   : {extra_in_target}")


# ------------------------------------------------------------
# Preview existing Silver
# ------------------------------------------------------------

display(
    existing_silver_payments_df
    .orderBy(F.col("last_updated").desc())
    .limit(10)
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 16, Finished, Available, Finished, False)

Existing Silver Payments table inspected.
------------------------------------------
Target table  : LH_Silver.dbo.silver_payments
Existing rows : 1500

Existing Silver Payments schema:
------------------------------------------
root
 |-- payment_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- claim_id: string (nullable = true)
 |-- payment_date: date (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- transaction_reference: string (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = true)


Duplicate payment_id keys : 0

Schema column comparison:
------------------------------------------
Missing in target : ['last_updated']
Extra in target   : []


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `last_updated` cannot be resolved. Did you mean one of the following? [`payment_date`, `claim_id`, `customer_id`, `payment_id`, `payment_status`].;
'Sort ['last_updated DESC NULLS LAST], true
+- SubqueryAlias spark_catalog.chimcobldhq2alid8pgm4sj9ccoj0c959h45ukr9dhr6ash5chh6u.silver_payments
   +- Relation spark_catalog.chimcobldhq2alid8pgm4sj9ccoj0c959h45ukr9dhr6ash5chh6u.silver_payments[payment_id#6251,policy_id#6252,customer_id#6253,claim_id#6254,payment_date#6255,payment_type#6256,payment_amount#6257,payment_method#6258,payment_status#6259,transaction_reference#6260,_silver_processed_ts#6261] parquet



### Step 8A — Align Silver Payments with the Incremental Schema

The existing Silver Payments table predates the introduction of the technical
`last_updated` watermark.

Target inspection confirmed that the existing Silver schema is compatible with
the new Silver-ready dataset except for the missing `last_updated` column.

To support reliable INSERT / UPDATE / NO-OP detection, the Silver table must
retain the source modification timestamp.

For existing historical Silver records, `payment_date` is used to initialize
`last_updated`. This mirrors the historical Bronze watermark initialization.

Future incremental records will carry their actual technical `last_updated`
value from Bronze.

This is a one-time schema alignment step. Existing Silver row counts and
business keys must remain unchanged.

In [15]:

# ============================================================
# STEP 8A - ALIGN SILVER PAYMENTS TECHNICAL WATERMARK
# ============================================================

from pyspark.sql import functions as F

silver_before_df = spark.table(CONFIG_TARGET_TABLE)

silver_before_count = silver_before_df.count()

print("Preparing Silver Payments schema alignment.")
print("------------------------------------------")
print(f"Silver rows before       : {silver_before_count}")
print(
    f"Existing last_updated    : "
    f"{'last_updated' in silver_before_df.columns}"
)


# ------------------------------------------------------------
# Add last_updated only when missing
# ------------------------------------------------------------

if "last_updated" not in silver_before_df.columns:

    silver_aligned_df = (
        silver_before_df
        .withColumn(
            "last_updated",
            F.to_timestamp(F.col("payment_date"))
        )
    )

    (
        silver_aligned_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(CONFIG_TARGET_TABLE)
    )

    print()
    print("last_updated added to Silver Payments.")

else:

    print()
    print("last_updated already exists.")
    print("No Silver schema modification required.")


# ------------------------------------------------------------
# Reload persisted target
# ------------------------------------------------------------

existing_silver_payments_df = spark.table(CONFIG_TARGET_TABLE)

silver_after_count = existing_silver_payments_df.count()


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert "last_updated" in existing_silver_payments_df.columns, \
    "Silver Payments last_updated was not created."

assert silver_after_count == silver_before_count, \
    "Silver Payments row count changed during schema alignment."


null_silver_watermarks = (
    existing_silver_payments_df
    .filter(F.col("last_updated").isNull())
    .count()
)

assert null_silver_watermarks == 0, \
    f"Found {null_silver_watermarks} NULL Silver watermarks."


duplicate_silver_keys = (
    existing_silver_payments_df
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_silver_keys == 0, \
    "Duplicate payment_id values detected after schema alignment."


print()
print("==========================================")
print(" SILVER PAYMENTS SCHEMA ALIGNMENT PASSED")
print("==========================================")
print(f"Rows before       : {silver_before_count}")
print(f"Rows after        : {silver_after_count}")
print(f"NULL watermarks   : {null_silver_watermarks}")
print(f"Duplicate keys    : {duplicate_silver_keys}")

print()
print("Aligned Silver Payments schema:")
existing_silver_payments_df.printSchema()


display(
    existing_silver_payments_df
    .select(
        "payment_id",
        "payment_date",
        "payment_amount",
        "payment_status",
        "last_updated",
        "_silver_processed_ts"
    )
    .orderBy(F.col("last_updated").desc())
    .limit(10)
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 17, Finished, Available, Finished, False)

Preparing Silver Payments schema alignment.
------------------------------------------
Silver rows before       : 1500
Existing last_updated    : False

last_updated added to Silver Payments.

 SILVER PAYMENTS SCHEMA ALIGNMENT PASSED
Rows before       : 1500
Rows after        : 1500
NULL watermarks   : 0
Duplicate keys    : 0

Aligned Silver Payments schema:
root
 |-- payment_id: string (nullable = true)
 |-- policy_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- claim_id: string (nullable = true)
 |-- payment_date: date (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- transaction_reference: string (nullable = true)
 |-- _silver_processed_ts: timestamp (nullable = true)
 |-- last_updated: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, ca65ed5a-e439-4b6e-bb55-e12da61fbb57)


## Step 9 — Classify Payments as INSERT, UPDATE, or NO-OP

Compare the Silver-ready Payments dataset with the existing Silver target using
`payment_id` as the business key.

Each incoming Payment is classified into one of three categories:

- **INSERT** — `payment_id` does not currently exist in Silver.
- **UPDATE** — `payment_id` exists, but one or more canonical business attributes differ.
- **NO-OP** — `payment_id` exists and all canonical business attributes are unchanged.

Change detection compares business attributes rather than technical processing
metadata. `_silver_processed_ts` is therefore excluded from the comparison.

The comparison is null-safe so that legitimate nullable fields such as `claim_id`
do not generate false updates.

This classification prevents unnecessary Delta writes and makes the incremental
pipeline idempotent.

No Silver data is modified in this step.

In [16]:
# ============================================================
# STEP 9 - CANONICAL INSERT / UPDATE / NO-OP DETECTION
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Refresh existing Silver target
# ------------------------------------------------------------

silver_current_df = spark.table(CONFIG_TARGET_TABLE)


# ------------------------------------------------------------
# Canonical business columns used for change detection
# ------------------------------------------------------------

comparison_columns = [
    "policy_id",
    "customer_id",
    "claim_id",
    "payment_date",
    "payment_type",
    "payment_amount",
    "payment_method",
    "payment_status",
    "transaction_reference"
]


# ------------------------------------------------------------
# Prepare source and target aliases
# ------------------------------------------------------------

src = silver_ready_payments_df.alias("src")
tgt = silver_current_df.alias("tgt")


# ------------------------------------------------------------
# Join incoming Payments to existing Silver
# ------------------------------------------------------------

payment_comparison_df = (
    src.join(
        tgt,
        F.col("src.payment_id") == F.col("tgt.payment_id"),
        "left"
    )
)


# ------------------------------------------------------------
# Determine whether any canonical business value changed
#
# eqNullSafe (<=> in SQL) prevents NULL vs NULL from being
# incorrectly classified as a change.
# ------------------------------------------------------------

business_changed_condition = None

for column_name in comparison_columns:

    column_changed = ~F.col(
        f"src.{column_name}"
    ).eqNullSafe(
        F.col(f"tgt.{column_name}")
    )

    if business_changed_condition is None:
        business_changed_condition = column_changed
    else:
        business_changed_condition = (
            business_changed_condition | column_changed
        )


# ------------------------------------------------------------
# Classify each incoming Payment
# ------------------------------------------------------------

payment_classification_df = (
    payment_comparison_df
    .withColumn(
        "_change_type",

        F.when(
            F.col("tgt.payment_id").isNull(),
            F.lit("INSERT")
        )

        .when(
            business_changed_condition,
            F.lit("UPDATE")
        )

        .otherwise(
            F.lit("NO_OP")
        )
    )
)


# ------------------------------------------------------------
# Produce clean source-shaped datasets
# ------------------------------------------------------------

insert_payments_df = (
    payment_classification_df
    .filter(F.col("_change_type") == "INSERT")
    .select("src.*")
)

update_payments_df = (
    payment_classification_df
    .filter(F.col("_change_type") == "UPDATE")
    .select("src.*")
)

noop_payments_df = (
    payment_classification_df
    .filter(F.col("_change_type") == "NO_OP")
    .select("src.*")
)


# ------------------------------------------------------------
# Counts
# ------------------------------------------------------------

insert_count = insert_payments_df.count()
update_count = update_payments_df.count()
noop_count = noop_payments_df.count()


# ------------------------------------------------------------
# Reconciliation
# ------------------------------------------------------------

assert (
    insert_count + update_count + noop_count
    == silver_ready_payments_count
), "Payments INSERT/UPDATE/NO-OP reconciliation failed."


print("Payments canonical change detection completed.")
print("-----------------------------------------------")
print(f"Silver-ready Payments : {silver_ready_payments_count}")
print(f"INSERT candidates     : {insert_count}")
print(f"UPDATE candidates     : {update_count}")
print(f"NO-OP Payments        : {noop_count}")
print(
    f"Reconciliation        : "
    f"{insert_count + update_count + noop_count}"
)


# ------------------------------------------------------------
# Classification summary
# ------------------------------------------------------------

display(
    payment_classification_df
    .groupBy("_change_type")
    .count()
    .orderBy("_change_type")
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 18, Finished, Available, Finished, False)

Payments canonical change detection completed.
-----------------------------------------------
Silver-ready Payments : 1499
INSERT candidates     : 0
UPDATE candidates     : 1499
NO-OP Payments        : 0
Reconciliation        : 1499


SynapseWidget(Synapse.DataFrame, a894af74-f71a-4473-bfca-b2714f29651a)

### Step 9A — Diagnose Canonical Payment Differences

The initial canonical comparison classified all 1,499 incoming Payments as
UPDATE candidates.

Because the Silver-ready and existing Silver datasets represent the same
historical Payments population, a 100% update rate is suspicious and must be
investigated before executing a Delta MERGE.

This diagnostic compares each canonical business attribute independently and
counts the number of rows where source and target differ.

The objective is to distinguish genuine business changes from representation
differences such as capitalization, whitespace, NULL handling, or historical
standardization.

No Silver data is modified in this step.

In [17]:

# ============================================================
# STEP 9A - COLUMN-LEVEL DIFFERENCE ANALYSIS
# ============================================================

from pyspark.sql import functions as F

diagnostic_columns = [
    "policy_id",
    "customer_id",
    "claim_id",
    "payment_date",
    "payment_type",
    "payment_amount",
    "payment_method",
    "payment_status",
    "transaction_reference"
]

src_diag = silver_ready_payments_df.alias("src")
tgt_diag = silver_current_df.alias("tgt")

matched_payments_df = (
    src_diag.join(
        tgt_diag,
        F.col("src.payment_id") == F.col("tgt.payment_id"),
        "inner"
    )
)

matched_count = matched_payments_df.count()

difference_results = []

for column_name in diagnostic_columns:

    different_count = (
        matched_payments_df
        .filter(
            ~F.col(f"src.{column_name}")
            .eqNullSafe(F.col(f"tgt.{column_name}"))
        )
        .count()
    )

    difference_results.append(
        (column_name, different_count)
    )


difference_df = spark.createDataFrame(
    difference_results,
    ["column_name", "different_rows"]
)

print("Payments column-level difference analysis completed.")
print("---------------------------------------------------")
print(f"Matched Payments : {matched_count}")

display(
    difference_df.orderBy(
        F.col("different_rows").desc()
    )
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 19, Finished, Available, Finished, False)

Payments column-level difference analysis completed.
---------------------------------------------------
Matched Payments : 1499


SynapseWidget(Synapse.DataFrame, dc8ebca0-c8af-4c08-a967-6fe8cf4bc3ab)


### Step 9B — Canonicalize Existing Silver Payments

Column-level diagnostics confirmed that the apparent mass UPDATE condition is
caused only by historical formatting differences in:

- `payment_type`
- `payment_method`

The incoming Silver-ready dataset uses standardized uppercase values, while
the existing Silver table contains mixed-case historical values.

To prevent formatting-only differences from generating false UPDATEs, the
existing Silver dataset is normalized using the same canonical transformation
rules as the incoming Payments.

Business attributes remain part of change detection; only their representation
is standardized before comparison.

No Silver data is modified in this step.

In [18]:
# ============================================================
# STEP 9B - CANONICALIZE EXISTING SILVER PAYMENTS
# ============================================================

canonical_silver_payments_df = (
    silver_current_df

    .withColumn(
        "payment_type",
        F.upper(F.trim(F.col("payment_type")))
    )

    .withColumn(
        "payment_method",
        F.upper(F.trim(F.col("payment_method")))
    )

    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status")))
    )
)


# ------------------------------------------------------------
# Verify previously detected differences are resolved
# ------------------------------------------------------------

canonical_check_df = (
    silver_ready_payments_df.alias("src")
    .join(
        canonical_silver_payments_df.alias("tgt"),
        F.col("src.payment_id") == F.col("tgt.payment_id"),
        "inner"
    )
)


payment_type_diff_after = (
    canonical_check_df
    .filter(
        ~F.col("src.payment_type")
        .eqNullSafe(F.col("tgt.payment_type"))
    )
    .count()
)

payment_method_diff_after = (
    canonical_check_df
    .filter(
        ~F.col("src.payment_method")
        .eqNullSafe(F.col("tgt.payment_method"))
    )
    .count()
)


print("Silver Payments canonicalization completed.")
print("--------------------------------------------")
print(f"payment_type differences   : {payment_type_diff_after}")
print(f"payment_method differences : {payment_method_diff_after}")

assert payment_type_diff_after == 0, \
    "payment_type differences remain after canonicalization."

assert payment_method_diff_after == 0, \
    "payment_method differences remain after canonicalization."

print()
print("PAYMENTS CANONICALIZATION VERIFIED.")

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 20, Finished, Available, Finished, False)

Silver Payments canonicalization completed.
--------------------------------------------
payment_type differences   : 0
payment_method differences : 0

PAYMENTS CANONICALIZATION VERIFIED.


### Step 9C — Reclassify Payments After Canonicalization

The historical Silver representation has now been normalized using the same
business rules as the incoming Silver-ready Payments.

The INSERT / UPDATE / NO-OP classification is rerun against the canonical
target.

This ensures only genuine business changes are classified as UPDATEs.

In [19]:
# ============================================================
# STEP 9C - RECLASSIFY PAYMENTS
# ============================================================

src = silver_ready_payments_df.alias("src")
tgt = canonical_silver_payments_df.alias("tgt")


payment_comparison_df = (
    src.join(
        tgt,
        F.col("src.payment_id") == F.col("tgt.payment_id"),
        "left"
    )
)


comparison_columns = [
    "policy_id",
    "customer_id",
    "claim_id",
    "payment_date",
    "payment_type",
    "payment_amount",
    "payment_method",
    "payment_status",
    "transaction_reference"
]


business_changed_condition = None

for column_name in comparison_columns:

    column_changed = (
        ~F.col(f"src.{column_name}")
        .eqNullSafe(F.col(f"tgt.{column_name}"))
    )

    business_changed_condition = (
        column_changed
        if business_changed_condition is None
        else business_changed_condition | column_changed
    )


payment_classification_df = (
    payment_comparison_df
    .withColumn(
        "_change_type",

        F.when(
            F.col("tgt.payment_id").isNull(),
            F.lit("INSERT")
        )

        .when(
            business_changed_condition,
            F.lit("UPDATE")
        )

        .otherwise(
            F.lit("NO_OP")
        )
    )
)


insert_payments_df = (
    payment_classification_df
    .filter(F.col("_change_type") == "INSERT")
    .select("src.*")
)

update_payments_df = (
    payment_classification_df
    .filter(F.col("_change_type") == "UPDATE")
    .select("src.*")
)

noop_payments_df = (
    payment_classification_df
    .filter(F.col("_change_type") == "NO_OP")
    .select("src.*")
)


insert_count = insert_payments_df.count()
update_count = update_payments_df.count()
noop_count = noop_payments_df.count()


assert (
    insert_count + update_count + noop_count
    == silver_ready_payments_count
), "Payments canonical reclassification failed."


print("Payments canonical reclassification completed.")
print("-----------------------------------------------")
print(f"Silver-ready Payments : {silver_ready_payments_count}")
print(f"INSERT candidates     : {insert_count}")
print(f"UPDATE candidates     : {update_count}")
print(f"NO-OP Payments        : {noop_count}")
print(
    f"Reconciliation        : "
    f"{insert_count + update_count + noop_count}"
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 21, Finished, Available, Finished, False)

Payments canonical reclassification completed.
-----------------------------------------------
Silver-ready Payments : 1499
INSERT candidates     : 0
UPDATE candidates     : 0
NO-OP Payments        : 1499
Reconciliation        : 1499



## Step 10 — Commit Batch Audit and Advance Watermark

The initial Payments incremental batch has completed successfully.

After validation, deduplication, canonical transformation, and change detection,
all 1,499 Silver-ready Payments were classified as NO-OP records. Therefore,
no Delta MERGE is required for this execution.

The pipeline now performs its control-plane commit:

1. Calculate the maximum successfully processed `last_updated` watermark.
2. Write the batch execution result to the ETL audit table.
3. Advance the PAYMENTS watermark in the ETL control table.
4. Verify that the persisted audit and watermark values are correct.

The watermark is advanced only after successful processing, preserving restart
safety and preventing data loss after a failed execution.

In [24]:
# ============================================================
# STEP 10 - AUDIT + WATERMARK COMMIT
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime


# ------------------------------------------------------------
# Calculate new watermark from successfully processed records
# ------------------------------------------------------------

NEW_PAYMENTS_WATERMARK = (
    silver_ready_payments_df
    .agg(F.max("last_updated").alias("max_watermark"))
    .first()["max_watermark"]
)

print("Payments watermark calculated.")
print("--------------------------------")
print(f"Previous watermark : {LAST_WATERMARK}")
print(f"New watermark      : {NEW_PAYMENTS_WATERMARK}")


assert NEW_PAYMENTS_WATERMARK is not None, \
    "Unable to calculate Payments watermark."

assert NEW_PAYMENTS_WATERMARK >= LAST_WATERMARK, \
    "Payments watermark moved backwards."


# ------------------------------------------------------------
# Audit information
# ------------------------------------------------------------

RUN_END_TS = datetime.utcnow()

source_count = incremental_payments_count
reject_count = rejected_payments_count

# These are already calculated in Step 9C
# insert_count
# update_count
# noop_count


# ------------------------------------------------------------
# Write SUCCESS audit
# ------------------------------------------------------------

audit_record = [
    (
        BATCH_ID,
        PIPELINE_NAME,
        SOURCE_NAME,
        RUN_START_TS,
        RUN_END_TS,
        source_count,
        insert_count,
        update_count,
        reject_count,
        "SUCCESS",
        None
    )
]


audit_columns = [
    "batch_id",
    "pipeline_name",
    "table_name",
    "start_time",
    "end_time",
    "source_count",
    "insert_count",
    "update_count",
    "reject_count",
    "status",
    "error_message"
]


from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

# ------------------------------------------------------------
# Advance control watermark
# ------------------------------------------------------------

from delta.tables import DeltaTable

control_delta = DeltaTable.forName(
    spark,
    CONTROL_TABLE
)

(
    control_delta.alias("tgt")
    .update(
        condition=(
            (F.col("source_name") == SOURCE_NAME) &
            (F.col("is_active") == True)
        ),
        set={
            "last_watermark": F.lit(NEW_PAYMENTS_WATERMARK),
            "_updated_ts": F.current_timestamp()
        }
    )
)

print("Payments watermark advanced.")


payments_audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("LH_Silver.dbo.etl_batch_audit")


print()
print("Payments SUCCESS audit written.")


# ------------------------------------------------------------
# Advance control watermark
# ------------------------------------------------------------

from delta.tables import DeltaTable

control_delta = DeltaTable.forName(
    spark,
    "LH_Silver.dbo.etl_control"
)

(
    control_delta.alias("tgt")
    .update(
        condition=(
            (F.col("tgt.source_name") == SOURCE_NAME) &
            (F.col("tgt.is_active") == True)
        ),
        set={
            "last_watermark": F.lit(NEW_PAYMENTS_WATERMARK),
            "updated_ts": F.current_timestamp()
        }
    )
)

print("Payments watermark advanced.")


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

payments_audit_check_df = (
    spark.table("LH_Silver.dbo.etl_batch_audit")
    .filter(F.col("batch_id") == BATCH_ID)
)

payments_control_check_df = (
    spark.table("LH_Silver.dbo.etl_control")
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
)


persisted_watermark = (
    payments_control_check_df
    .select("last_watermark")
    .first()["last_watermark"]
)


assert payments_audit_check_df.count() == 1, \
    "Expected exactly one Payments audit record."

assert persisted_watermark == NEW_PAYMENTS_WATERMARK, \
    "Persisted Payments watermark does not match calculated watermark."


print()
print("==========================================")
print(" PAYMENTS INITIAL INCREMENTAL RUN PASSED")
print("==========================================")
print(f"Bronze incremental rows : {source_count}")
print(f"Valid Payments          : {valid_payments_count}")
print(f"Rejected Payments       : {reject_count}")
print(f"Processed INSERTS       : {insert_count}")
print(f"Processed UPDATES       : {update_count}")
print(f"NO-OP Payments          : {noop_count}")
print(f"Final watermark         : {NEW_PAYMENTS_WATERMARK}")

display(payments_audit_check_df)
display(payments_control_check_df)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 26, Finished, Available, Finished, False)

Payments watermark advanced.


In [25]:
# ============================================================
# STEP 10B - VERIFY PAYMENTS AUDIT + WATERMARK COMMIT
# ============================================================

# ------------------------------------------------------------
# Verify audit row already written for this batch
# ------------------------------------------------------------

payments_audit_check_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

payments_audit_count = payments_audit_check_df.count()

assert payments_audit_count == 1, (
    f"Expected exactly one Payments audit record for batch "
    f"{BATCH_ID}, found {payments_audit_count}."
)

payments_audit_row = payments_audit_check_df.first()

assert payments_audit_row["status"] == "SUCCESS", \
    "Payments audit status is not SUCCESS."

assert payments_audit_row["source_count"] == incremental_payments_count, \
    "Payments audit source_count mismatch."

assert payments_audit_row["insert_count"] == insert_count, \
    "Payments audit insert_count mismatch."

assert payments_audit_row["update_count"] == update_count, \
    "Payments audit update_count mismatch."

assert payments_audit_row["reject_count"] == rejected_payments_count, \
    "Payments audit reject_count mismatch."


# ------------------------------------------------------------
# Verify persisted control watermark
# ------------------------------------------------------------

payments_control_check_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
)

payments_control_row = payments_control_check_df.first()

persisted_payments_watermark = (
    payments_control_row["last_watermark"]
)

assert (
    persisted_payments_watermark == NEW_PAYMENTS_WATERMARK
), (
    "Persisted PAYMENTS watermark does not match "
    "the successfully processed watermark."
)


print()
print("==========================================")
print(" PAYMENTS INITIAL INCREMENTAL RUN PASSED")
print("==========================================")
print(f"Bronze incremental rows : {incremental_payments_count}")
print(f"Valid Payments          : {valid_payments_count}")
print(f"Rejected Payments       : {rejected_payments_count}")
print(f"Unique Payments         : {unique_payments_count}")
print(f"Duplicate versions      : {duplicate_versions_count}")
print(f"Processed INSERTS       : {insert_count}")
print(f"Processed UPDATES       : {update_count}")
print(f"NO-OP Payments          : {noop_count}")
print(f"Final watermark         : {persisted_payments_watermark}")
print(f"Audit status            : {payments_audit_row['status']}")

display(payments_audit_check_df)
display(payments_control_check_df)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 27, Finished, Available, Finished, False)

AssertionError: Expected exactly one Payments audit record for batch e1ed9e2a-52d3-48cd-bff8-11cbdf960abe, found 3.

### Step 10C — Repair Duplicate Batch Audit Records

During recovery from the Payments watermark-update error, the SUCCESS audit
write completed before the later control-table operation failed.

Rerunning the cell therefore created multiple audit records for the same
`batch_id`.

This step repairs only the audit metadata:

1. Remove duplicate audit rows for the current Payments batch.
2. Recreate exactly one canonical SUCCESS audit record.
3. Verify that the batch has exactly one audit record.
4. Leave the already-correct Payments watermark unchanged.

No Bronze or Silver business data is modified.

In [26]:
# ============================================================
# STEP 10C - REPAIR DUPLICATE PAYMENTS AUDIT RECORDS
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

# ------------------------------------------------------------
# 1. Inspect duplicate count
# ------------------------------------------------------------

duplicate_batch_audit_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

duplicate_batch_count = duplicate_batch_audit_df.count()

print("Payments audit repair started.")
print("------------------------------------------")
print(f"Batch ID             : {BATCH_ID}")
print(f"Existing audit rows  : {duplicate_batch_count}")

display(duplicate_batch_audit_df)


# ------------------------------------------------------------
# 2. Remove all audit rows for THIS batch only
# ------------------------------------------------------------

spark.sql(f"""
    DELETE FROM {AUDIT_TABLE}
    WHERE batch_id = '{BATCH_ID}'
""")

remaining_count = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
    .count()
)

assert remaining_count == 0, \
    "Unable to remove duplicate Payments audit records."


# ------------------------------------------------------------
# 3. Recreate one canonical audit record
# ------------------------------------------------------------

payments_audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("source_count", LongType(), False),
    StructField("insert_count", LongType(), False),
    StructField("update_count", LongType(), False),
    StructField("reject_count", LongType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

repair_end_ts = datetime.now()

canonical_audit_record = [(
    BATCH_ID,
    PIPELINE_NAME,
    SOURCE_NAME,
    RUN_START_TS,
    repair_end_ts,
    int(incremental_payments_count),
    int(insert_count),
    int(update_count),
    int(rejected_payments_count),
    "SUCCESS",
    None
)]

canonical_audit_df = spark.createDataFrame(
    canonical_audit_record,
    schema=payments_audit_schema
)

(
    canonical_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)


# ------------------------------------------------------------
# 4. Final audit verification
# ------------------------------------------------------------

payments_audit_check_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)

final_audit_count = payments_audit_check_df.count()

assert final_audit_count == 1, \
    f"Expected exactly one repaired audit row, found {final_audit_count}."


# ------------------------------------------------------------
# 5. Verify watermark remains correct
# ------------------------------------------------------------

payments_control_check_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
)

persisted_payments_watermark = (
    payments_control_check_df
    .first()["last_watermark"]
)

assert persisted_payments_watermark == NEW_PAYMENTS_WATERMARK, \
    "Payments watermark changed during audit repair."


print()
print("==========================================")
print(" PAYMENTS AUDIT REPAIR PASSED")
print("==========================================")
print(f"Audit rows          : {final_audit_count}")
print(f"Source count        : {incremental_payments_count}")
print(f"INSERT count        : {insert_count}")
print(f"UPDATE count        : {update_count}")
print(f"Reject count        : {rejected_payments_count}")
print(f"Status              : SUCCESS")
print(f"Persisted watermark : {persisted_payments_watermark}")

display(payments_audit_check_df)
display(payments_control_check_df)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 28, Finished, Available, Finished, False)

Payments audit repair started.
------------------------------------------
Batch ID             : e1ed9e2a-52d3-48cd-bff8-11cbdf960abe
Existing audit rows  : 3


SynapseWidget(Synapse.DataFrame, 40d98186-2c84-4a4b-88a6-04bf7f94cc55)


 PAYMENTS AUDIT REPAIR PASSED
Audit rows          : 1
Source count        : 1501
INSERT count        : 0
UPDATE count        : 0
Reject count        : 1
Status              : SUCCESS
Persisted watermark : 2027-10-08 00:00:00


SynapseWidget(Synapse.DataFrame, 1aadba54-7ecf-4b3f-98db-14a6a6becf70)

SynapseWidget(Synapse.DataFrame, 7fa650d2-55d3-448d-9bb1-2ca6761a8fd7)


## Step 11 — Validate Restart and Idempotency

The successful Payments batch advanced the persisted watermark to
`2027-10-08`.

This step simulates a pipeline restart by reading the persisted watermark
directly from the ETL control table and applying the incremental extraction
condition to Bronze Payments again.

A correctly implemented incremental pipeline must:

- Select only records newer than the committed watermark.
- Avoid reprocessing previously completed Payments.
- Avoid duplicate INSERT operations.
- Avoid unnecessary UPDATE operations.
- Leave the Silver table unchanged when no new data is available.

This validates restart safety and idempotent incremental processing.

No Bronze or Silver data is modified in this step.

In [27]:
# ============================================================
# STEP 11 - PAYMENTS RESTART / IDEMPOTENCY TEST
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Read CURRENT persisted watermark
# ------------------------------------------------------------

restart_control_row = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
    .first()
)

RESTART_WATERMARK = restart_control_row["last_watermark"]


# ------------------------------------------------------------
# Reload Bronze Payments
# ------------------------------------------------------------

restart_bronze_df = spark.table(CONFIG_SOURCE_TABLE)

restart_bronze_count = restart_bronze_df.count()


# ------------------------------------------------------------
# Apply incremental condition
# IMPORTANT: strictly greater than committed watermark
# ------------------------------------------------------------

restart_incremental_df = (
    restart_bronze_df
    .withColumn(
        "_restart_watermark_ts",
        F.to_timestamp(F.col(CONFIG_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_restart_watermark_ts") >
        F.lit(RESTART_WATERMARK)
    )
)

restart_incremental_count = restart_incremental_df.count()


# ------------------------------------------------------------
# Silver state
# ------------------------------------------------------------

restart_silver_count = (
    spark.table(CONFIG_TARGET_TABLE)
    .count()
)


print("Payments restart test completed.")
print("------------------------------------------")
print(f"Bronze Payments rows : {restart_bronze_count}")
print(f"Stored watermark     : {RESTART_WATERMARK}")
print(f"Incremental rows     : {restart_incremental_count}")
print(f"Silver Payments rows : {restart_silver_count}")


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert restart_incremental_count == 0, (
    f"Restart selected {restart_incremental_count} "
    "previously processed Payments."
)

assert restart_silver_count == 1500, (
    "Silver Payments row count changed unexpectedly."
)


print()
print("==========================================")
print(" PAYMENTS RESTART / IDEMPOTENCY TEST PASSED")
print("==========================================")
print("No previously processed Payments were selected.")
print("No duplicate INSERT occurred.")
print("No unnecessary UPDATE occurred.")
print("Silver state remained unchanged.")

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 29, Finished, Available, Finished, False)

Payments restart test completed.
------------------------------------------
Bronze Payments rows : 1501
Stored watermark     : 2027-10-08 00:00:00
Incremental rows     : 0
Silver Payments rows : 1500

 PAYMENTS RESTART / IDEMPOTENCY TEST PASSED
No previously processed Payments were selected.
No duplicate INSERT occurred.
No unnecessary UPDATE occurred.
Silver state remained unchanged.



## Step 12 — Controlled Incremental INSERT and UPDATE Test

The baseline Payments incremental pipeline and restart behavior have been
successfully validated.

This final test introduces two controlled Bronze records with timestamps later
than the committed watermark:

- **UPDATE scenario** — create a newer version of an existing `payment_id`
  with a modified business attribute.
- **INSERT scenario** — create a completely new `payment_id`.

The test validates the complete production incremental path:

**Bronze → Watermark Extraction → Validation → Deduplication → Canonicalization
→ INSERT / UPDATE Classification → Delta MERGE → Audit → Watermark Commit
→ Restart / Idempotency**

The controlled records are first created in memory for inspection.

Nothing is written to Bronze until the records are explicitly verified.

In [29]:

# ============================================================
# STEP 12A - CREATE CONTROLLED PAYMENT TEST RECORDS
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Select an existing Payment for UPDATE
# ------------------------------------------------------------

existing_payment_row = (
    spark.table(CONFIG_SOURCE_TABLE)
    .filter(F.col("payment_amount").cast("double") > 0)
    .orderBy(F.col("last_updated").desc())
    .first()
)

assert existing_payment_row is not None, \
    "Unable to find an existing Payment for controlled UPDATE."


UPDATE_PAYMENT_ID = existing_payment_row["payment_id"]
INSERT_PAYMENT_ID = "PAY999901"


print("Existing Payment selected for controlled UPDATE.")
print("-----------------------------------------------")
print(f"Payment ID : {UPDATE_PAYMENT_ID}")


# ------------------------------------------------------------
# Build UPDATE version
# ------------------------------------------------------------

update_payment_df = (
    spark.table(CONFIG_SOURCE_TABLE)
    .filter(F.col("payment_id") == UPDATE_PAYMENT_ID)
    .orderBy(F.col("last_updated").desc())
    .limit(1)

    # Make a visible business change
    .withColumn(
        "payment_amount",
        (
            F.col("payment_amount").cast("double") + F.lit(125.00)
        ).cast("string")
    )

    # Must be later than current committed watermark
    .withColumn(
        "last_updated",
        F.lit("2027-10-09").cast("date")
    )
)


# ------------------------------------------------------------
# Build completely new INSERT record
# ------------------------------------------------------------

insert_payment_df = (
    update_payment_df

    .withColumn(
        "payment_id",
        F.lit(INSERT_PAYMENT_ID)
    )

    .withColumn(
        "transaction_reference",
        F.lit("TXN-TEST-PAY999901")
    )

    .withColumn(
        "payment_amount",
        F.lit("2500.00")
    )

    .withColumn(
        "payment_status",
        F.lit("Completed")
    )

    .withColumn(
        "last_updated",
        F.lit("2027-10-10").cast("date")
    )
)


controlled_payments_test_df = (
    update_payment_df
    .unionByName(insert_payment_df)
)


controlled_test_count = controlled_payments_test_df.count()

assert controlled_test_count == 2, \
    "Expected exactly two controlled Payments test records."


print()
print("Controlled Payments test records created.")
print("-----------------------------------------------")
print(f"UPDATE Payment : {UPDATE_PAYMENT_ID}")
print(f"INSERT Payment : {INSERT_PAYMENT_ID}")
print(f"Test records   : {controlled_test_count}")
print()
print("Nothing has been written to Bronze yet.")


display(
    controlled_payments_test_df.select(
        "payment_id",
        "policy_id",
        "customer_id",
        "claim_id",
        "payment_type",
        "payment_amount",
        "payment_method",
        "payment_status",
        "transaction_reference",
        "last_updated"
    )
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 31, Finished, Available, Finished, False)

Existing Payment selected for controlled UPDATE.
-----------------------------------------------
Payment ID : PAY0000114

Controlled Payments test records created.
-----------------------------------------------
UPDATE Payment : PAY0000114
INSERT Payment : PAY999901
Test records   : 2

Nothing has been written to Bronze yet.


SynapseWidget(Synapse.DataFrame, ff6b5e69-a7d7-4c74-bb9e-ade4845b826f)


### Step 12B — Append Controlled Test Records to Bronze

The two controlled Payment records have been validated in memory.

This step appends them to the Bronze Payments Delta table to simulate a new
incremental source delivery:

- One newer version of an existing Payment for UPDATE processing.
- One new Payment for INSERT processing.

After the append, Bronze row count must increase by exactly two records.

The committed watermark remains unchanged until the incremental batch
successfully completes.

In [30]:

# ============================================================
# STEP 12B - APPEND CONTROLLED PAYMENT RECORDS TO BRONZE
# ============================================================

bronze_before_count = (
    spark.table(CONFIG_SOURCE_TABLE)
    .count()
)

# Align test records exactly to Bronze schema
bronze_schema = spark.table(CONFIG_SOURCE_TABLE).schema

controlled_bronze_df = (
    spark.createDataFrame(
        controlled_payments_test_df.rdd,
        schema=bronze_schema
    )
)

# Append exactly two controlled records
(
    controlled_bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(CONFIG_SOURCE_TABLE)
)

bronze_after_count = (
    spark.table(CONFIG_SOURCE_TABLE)
    .count()
)

net_increase = bronze_after_count - bronze_before_count

assert net_increase == 2, (
    f"Expected Bronze to increase by 2 rows, "
    f"but increased by {net_increase}."
)

print("Controlled Payments records appended to Bronze.")
print("-----------------------------------------------")
print(f"Bronze rows before : {bronze_before_count}")
print(f"Bronze rows after  : {bronze_after_count}")
print(f"Net increase       : {net_increase}")
print()
print(f"UPDATE test Payment : {UPDATE_PAYMENT_ID}")
print(f"INSERT test Payment : {INSERT_PAYMENT_ID}")

display(
    spark.table(CONFIG_SOURCE_TABLE)
    .filter(
        F.col("payment_id").isin(
            UPDATE_PAYMENT_ID,
            INSERT_PAYMENT_ID
        )
    )
    .select(
        "payment_id",
        "payment_type",
        "payment_amount",
        "payment_method",
        "payment_status",
        "transaction_reference",
        "last_updated"
    )
    .orderBy(
        "payment_id",
        F.col("last_updated").desc()
    )
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 32, Finished, Available, Finished, False)

Controlled Payments records appended to Bronze.
-----------------------------------------------
Bronze rows before : 1501
Bronze rows after  : 1503
Net increase       : 2

UPDATE test Payment : PAY0000114
INSERT test Payment : PAY999901


SynapseWidget(Synapse.DataFrame, ecc31f92-e46b-4cfa-858b-d5e3f760b991)

### Step 12C — Validate Controlled Incremental Classification

This step re-runs the incremental selection logic using the committed Payments
watermark.

The two controlled Bronze records have timestamps later than the persisted
watermark and should therefore be selected as the new incremental batch.

Expected classification:

- **1 INSERT** — the new Payment does not exist in Silver.
- **1 UPDATE** — the existing Payment has a newer changed version.
- **0 NO-OP** — both selected records represent genuine changes.

No Silver data is modified in this step.

In [31]:
# ============================================================
# STEP 12C - CONTROLLED PAYMENTS INCREMENTAL CLASSIFICATION
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Read persisted watermark from control table
# ------------------------------------------------------------

control_row = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
    .first()
)

CONTROLLED_WATERMARK = control_row["last_watermark"]

print("Controlled Payments batch started.")
print("------------------------------------------")
print(f"Stored watermark : {CONTROLLED_WATERMARK}")


# ------------------------------------------------------------
# Incremental extraction
# ------------------------------------------------------------

controlled_incremental_df = (
    spark.table(CONFIG_SOURCE_TABLE)
    .withColumn(
        "_watermark_ts",
        F.to_timestamp(F.col(CONFIG_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_watermark_ts") >
        F.lit(CONTROLLED_WATERMARK)
    )
)

incremental_count = controlled_incremental_df.count()

assert incremental_count == 2, (
    f"Expected exactly 2 incremental Payments, "
    f"found {incremental_count}."
)


# ------------------------------------------------------------
# Prepare canonical comparison representation
# ------------------------------------------------------------

comparison_columns = [
    "policy_id",
    "customer_id",
    "claim_id",
    "payment_date",
    "payment_type",
    "payment_amount",
    "payment_method",
    "payment_status",
    "transaction_reference"
]


incoming_df = (
    controlled_incremental_df

    .withColumn(
        "payment_date",
        F.to_date("payment_date")
    )

    .withColumn(
        "payment_amount",
        F.col("payment_amount").cast("double")
    )

    .withColumn(
        "payment_type",
        F.upper(F.trim("payment_type"))
    )

    .withColumn(
        "payment_method",
        F.upper(F.trim("payment_method"))
    )

    .withColumn(
        "payment_status",
        F.upper(F.trim("payment_status"))
    )

    .withColumn(
        "last_updated",
        F.to_timestamp("last_updated")
    )
)


silver_compare_df = (
    spark.table(CONFIG_TARGET_TABLE)
    .select(
        "payment_id",
        *comparison_columns
    )
)


# ------------------------------------------------------------
# Join incoming Payments to Silver
# ------------------------------------------------------------

joined_df = (
    incoming_df.alias("src")
    .join(
        silver_compare_df.alias("tgt"),
        F.col("src.payment_id") == F.col("tgt.payment_id"),
        "left"
    )
)


# ------------------------------------------------------------
# Null-safe business-column comparison
# ------------------------------------------------------------

different_condition = None

for column_name in comparison_columns:

    column_difference = ~(
        F.col(f"src.{column_name}")
        .eqNullSafe(F.col(f"tgt.{column_name}"))
    )

    different_condition = (
        column_difference
        if different_condition is None
        else different_condition | column_difference
    )


# ------------------------------------------------------------
# Classify INSERT / UPDATE / NO-OP
# ------------------------------------------------------------

classified_df = (
    joined_df
    .withColumn(
        "_change_type",
        F.when(
            F.col("tgt.payment_id").isNull(),
            F.lit("INSERT")
        )
        .when(
            different_condition,
            F.lit("UPDATE")
        )
        .otherwise(
            F.lit("NO-OP")
        )
    )
)


insert_count = (
    classified_df
    .filter(F.col("_change_type") == "INSERT")
    .count()
)

update_count = (
    classified_df
    .filter(F.col("_change_type") == "UPDATE")
    .count()
)

noop_count = (
    classified_df
    .filter(F.col("_change_type") == "NO-OP")
    .count()
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert insert_count == 1, (
    f"Expected 1 INSERT, found {insert_count}."
)

assert update_count == 1, (
    f"Expected 1 UPDATE, found {update_count}."
)

assert noop_count == 0, (
    f"Expected 0 NO-OP Payments, found {noop_count}."
)

assert (
    insert_count + update_count + noop_count
    == incremental_count
), "Controlled Payments reconciliation failed."


print()
print("==========================================")
print(" CONTROLLED PAYMENTS CLASSIFICATION PASSED")
print("==========================================")
print(f"Incremental records : {incremental_count}")
print(f"INSERT candidates   : {insert_count}")
print(f"UPDATE candidates   : {update_count}")
print(f"NO-OP Payments      : {noop_count}")
print(f"Reconciliation      : {insert_count + update_count + noop_count}")

display(
    classified_df.select(
        F.col("src.payment_id").alias("payment_id"),
        F.col("src.payment_amount").alias("incoming_amount"),
        F.col("src.payment_status").alias("incoming_status"),
        F.col("src.last_updated").alias("incoming_last_updated"),
        "_change_type"
    )
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 33, Finished, Available, Finished, False)

Controlled Payments batch started.
------------------------------------------
Stored watermark : 2027-10-08 00:00:00

 CONTROLLED PAYMENTS CLASSIFICATION PASSED
Incremental records : 2
INSERT candidates   : 1
UPDATE candidates   : 1
NO-OP Payments      : 0
Reconciliation      : 2


SynapseWidget(Synapse.DataFrame, b759940c-e4fb-4529-8429-f92c2bbfca99)

### Step 12D — Execute Controlled Payments Delta MERGE

The controlled incremental batch has been successfully classified:

- **1 INSERT** — a new Payment that does not exist in Silver.
- **1 UPDATE** — an existing Payment with changed business data.
- **0 NO-OP records**.

This step applies those changes to the Silver Payments Delta table using an
idempotent Delta MERGE keyed by `payment_id`.

Expected result:

- The existing Payment is **updated in place**, not duplicated.
- The new Payment is **inserted exactly once**.
- Silver row count increases by exactly **one**.
- `payment_id` remains unique in Silver.
- The control watermark is **not advanced yet**.

Audit and watermark commit occur only after the MERGE is validated successfully.

In [32]:
# ============================================================
# STEP 12D - CONTROLLED PAYMENTS DELTA MERGE
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F


# ------------------------------------------------------------
# Build MERGE source
# Only INSERT + UPDATE records participate
# ------------------------------------------------------------

merge_source_df = (
    classified_df
    .filter(F.col("_change_type").isin("INSERT", "UPDATE"))
    .select(
        F.col("src.payment_id").alias("payment_id"),
        F.col("src.policy_id").alias("policy_id"),
        F.col("src.customer_id").alias("customer_id"),
        F.col("src.claim_id").alias("claim_id"),
        F.col("src.payment_date").alias("payment_date"),
        F.col("src.payment_type").alias("payment_type"),
        F.col("src.payment_amount").alias("payment_amount"),
        F.col("src.payment_method").alias("payment_method"),
        F.col("src.payment_status").alias("payment_status"),
        F.col("src.transaction_reference").alias("transaction_reference"),
        F.col("src.last_updated").alias("last_updated")
    )
    .withColumn(
        "_silver_processed_ts",
        F.current_timestamp()
    )
)


# ------------------------------------------------------------
# Pre-MERGE checks
# ------------------------------------------------------------

silver_rows_before = spark.table(CONFIG_TARGET_TABLE).count()

merge_source_count = merge_source_df.count()

assert merge_source_count == 2, (
    f"Expected 2 MERGE records, found {merge_source_count}."
)

print("Preparing controlled Payments Delta MERGE.")
print("------------------------------------------")
print(f"Silver rows before : {silver_rows_before}")
print(f"INSERT records     : {insert_count}")
print(f"UPDATE records     : {update_count}")


# ------------------------------------------------------------
# Execute Delta MERGE
# ------------------------------------------------------------

payments_delta = DeltaTable.forName(
    spark,
    CONFIG_TARGET_TABLE
)

(
    payments_delta.alias("tgt")
    .merge(
        merge_source_df.alias("src"),
        "tgt.payment_id = src.payment_id"
    )

    .whenMatchedUpdate(
        set={
            "policy_id": "src.policy_id",
            "customer_id": "src.customer_id",
            "claim_id": "src.claim_id",
            "payment_date": "src.payment_date",
            "payment_type": "src.payment_type",
            "payment_amount": "src.payment_amount",
            "payment_method": "src.payment_method",
            "payment_status": "src.payment_status",
            "transaction_reference": "src.transaction_reference",
            "last_updated": "src.last_updated",
            "_silver_processed_ts": "src._silver_processed_ts"
        }
    )

    .whenNotMatchedInsert(
        values={
            "payment_id": "src.payment_id",
            "policy_id": "src.policy_id",
            "customer_id": "src.customer_id",
            "claim_id": "src.claim_id",
            "payment_date": "src.payment_date",
            "payment_type": "src.payment_type",
            "payment_amount": "src.payment_amount",
            "payment_method": "src.payment_method",
            "payment_status": "src.payment_status",
            "transaction_reference": "src.transaction_reference",
            "last_updated": "src.last_updated",
            "_silver_processed_ts": "src._silver_processed_ts"
        }
    )

    .execute()
)


# ------------------------------------------------------------
# Post-MERGE validation
# ------------------------------------------------------------

silver_after_df = spark.table(CONFIG_TARGET_TABLE)

silver_rows_after = silver_after_df.count()

net_row_increase = silver_rows_after - silver_rows_before


# Must increase only by INSERT count.
assert net_row_increase == 1, (
    f"Expected Silver to increase by 1 row, "
    f"but increased by {net_row_increase}."
)


# ------------------------------------------------------------
# Verify both test Payments
# ------------------------------------------------------------

test_results_df = (
    silver_after_df
    .filter(
        F.col("payment_id").isin(
            UPDATE_PAYMENT_ID,
            INSERT_PAYMENT_ID
        )
    )
)

test_result_count = test_results_df.count()

assert test_result_count == 2, (
    f"Expected exactly 2 controlled Payments in Silver, "
    f"found {test_result_count}."
)


# ------------------------------------------------------------
# Verify UPDATE did not create duplicate
# ------------------------------------------------------------

update_key_count = (
    silver_after_df
    .filter(F.col("payment_id") == UPDATE_PAYMENT_ID)
    .count()
)

assert update_key_count == 1, (
    f"UPDATE Payment {UPDATE_PAYMENT_ID} "
    f"appears {update_key_count} times in Silver."
)


# ------------------------------------------------------------
# Verify INSERT exists once
# ------------------------------------------------------------

insert_key_count = (
    silver_after_df
    .filter(F.col("payment_id") == INSERT_PAYMENT_ID)
    .count()
)

assert insert_key_count == 1, (
    f"INSERT Payment {INSERT_PAYMENT_ID} "
    f"appears {insert_key_count} times in Silver."
)


print()
print("Controlled Payments Delta MERGE completed.")
print()

print("==========================================")
print(" CONTROLLED PAYMENTS DELTA MERGE PASSED")
print("==========================================")
print(f"Silver rows before : {silver_rows_before}")
print(f"Silver rows after  : {silver_rows_after}")
print(f"Net row increase   : {net_row_increase}")
print(f"Processed INSERTS  : {insert_count}")
print(f"Processed UPDATES  : {update_count}")

display(
    test_results_df.select(
        "payment_id",
        "policy_id",
        "payment_type",
        "payment_amount",
        "payment_method",
        "payment_status",
        "transaction_reference",
        "last_updated",
        "_silver_processed_ts"
    )
    .orderBy("payment_id")
)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 34, Finished, Available, Finished, False)

Preparing controlled Payments Delta MERGE.
------------------------------------------
Silver rows before : 1500
INSERT records     : 1
UPDATE records     : 1

Controlled Payments Delta MERGE completed.

 CONTROLLED PAYMENTS DELTA MERGE PASSED
Silver rows before : 1500
Silver rows after  : 1501
Net row increase   : 1
Processed INSERTS  : 1
Processed UPDATES  : 1


SynapseWidget(Synapse.DataFrame, 2ece9de4-d736-4a42-a6a1-66084237934c)

### Step 12E — Commit Batch Audit and Advance Watermark

The controlled Payments Delta MERGE completed successfully.

This step commits the batch by:

- Calculating the maximum successfully processed `last_updated` value.
- Writing one SUCCESS record to the ETL batch audit table.
- Advancing the Payments control watermark only after the Silver MERGE succeeded.
- Verifying the persisted control state.

For this controlled batch, the watermark should advance:

**2027-10-08 → 2027-10-10**

This ordering ensures that a failed Silver write cannot cause the control
watermark to advance past unprocessed data.

In [33]:

# ============================================================
# STEP 12E - PAYMENTS AUDIT + WATERMARK COMMIT
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)
from delta.tables import DeltaTable
from datetime import datetime
import uuid


# ------------------------------------------------------------
# Batch metadata
# ------------------------------------------------------------

CONTROLLED_BATCH_ID = str(uuid.uuid4())
CONTROLLED_BATCH_START = datetime.now()

NEW_PAYMENTS_WATERMARK = (
    controlled_incremental_df
    .agg(F.max("_watermark_ts").alias("max_watermark"))
    .first()["max_watermark"]
)

assert NEW_PAYMENTS_WATERMARK is not None, \
    "Unable to calculate Payments watermark."

assert NEW_PAYMENTS_WATERMARK > CONTROLLED_WATERMARK, (
    "New Payments watermark did not advance."
)


print("Controlled Payments watermark calculated.")
print("------------------------------------------")
print(f"Previous watermark : {CONTROLLED_WATERMARK}")
print(f"New watermark      : {NEW_PAYMENTS_WATERMARK}")


# ------------------------------------------------------------
# Write SUCCESS audit record
# ------------------------------------------------------------

CONTROLLED_BATCH_END = datetime.now()

audit_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("source_count", LongType(), False),
    StructField("insert_count", LongType(), False),
    StructField("update_count", LongType(), False),
    StructField("reject_count", LongType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

controlled_audit_df = spark.createDataFrame(
    [(
        CONTROLLED_BATCH_ID,
        PIPELINE_NAME,
        SOURCE_NAME,
        CONTROLLED_BATCH_START,
        CONTROLLED_BATCH_END,
        int(incremental_count),
        int(insert_count),
        int(update_count),
        0,
        "SUCCESS",
        None
    )],
    schema=audit_schema
)

controlled_audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(AUDIT_TABLE)

print()
print("Controlled Payments SUCCESS audit written.")


# ------------------------------------------------------------
# Advance watermark AFTER successful MERGE + audit
# ------------------------------------------------------------

control_delta = DeltaTable.forName(
    spark,
    CONTROL_TABLE
)

(
    control_delta.alias("tgt")
    .update(
        condition=(
            (F.col("source_name") == SOURCE_NAME) &
            (F.col("is_active") == True)
        ),
        set={
            "last_watermark": F.lit(NEW_PAYMENTS_WATERMARK),
            "_updated_ts": F.current_timestamp()
        }
    )
)

print("Payments watermark advanced.")


# ------------------------------------------------------------
# Verify audit record
# ------------------------------------------------------------

payments_audit_check_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("batch_id") == CONTROLLED_BATCH_ID)
)

audit_count = payments_audit_check_df.count()

assert audit_count == 1, (
    f"Expected exactly one audit record, found {audit_count}."
)


# ------------------------------------------------------------
# Verify persisted watermark
# ------------------------------------------------------------

payments_control_check_df = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
)

persisted_payments_watermark = (
    payments_control_check_df
    .first()["last_watermark"]
)

assert persisted_payments_watermark == NEW_PAYMENTS_WATERMARK, (
    "Persisted Payments watermark does not match "
    "the committed batch watermark."
)


print()
print("==========================================")
print(" PAYMENTS AUDIT + WATERMARK COMMIT PASSED")
print("==========================================")
print(f"Batch ID          : {CONTROLLED_BATCH_ID}")
print(f"Source records    : {incremental_count}")
print(f"INSERT records    : {insert_count}")
print(f"UPDATE records    : {update_count}")
print(f"Rejected records  : 0")
print(f"Final watermark   : {persisted_payments_watermark}")
print(f"Status            : SUCCESS")

display(payments_audit_check_df)
display(payments_control_check_df)

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 35, Finished, Available, Finished, False)

Controlled Payments watermark calculated.
------------------------------------------
Previous watermark : 2027-10-08 00:00:00
New watermark      : 2027-10-10 00:00:00

Controlled Payments SUCCESS audit written.
Payments watermark advanced.

 PAYMENTS AUDIT + WATERMARK COMMIT PASSED
Batch ID          : 2fffe787-fcde-4d65-8cc3-7e806c0a7c64
Source records    : 2
INSERT records    : 1
UPDATE records    : 1
Rejected records  : 0
Final watermark   : 2027-10-10 00:00:00
Status            : SUCCESS


SynapseWidget(Synapse.DataFrame, 3bfbee2f-4bf2-4e6a-a216-65f971cb6ea4)

SynapseWidget(Synapse.DataFrame, 90e297e7-dfe3-4ea9-b743-8c8b175715aa)

### Step 12F — Final Restart and Idempotency Validation

The controlled Payments batch has been successfully committed with a persisted
watermark of `2027-10-10`.

This final test simulates restarting the Payments pipeline after the successful
batch commit.

The pipeline must:

- Read the committed watermark from the ETL control table.
- Select only Bronze records with `last_updated` greater than the watermark.
- Return zero incremental records.
- Avoid duplicate INSERT processing.
- Avoid repeating the UPDATE.
- Leave the Silver Payments table unchanged.

A zero-row incremental result proves that the Payments pipeline is restart-safe
and idempotent after a successful INSERT/UPDATE batch.

In [34]:
# ============================================================
# STEP 12F - FINAL PAYMENTS RESTART / IDEMPOTENCY TEST
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Read persisted watermark
# ------------------------------------------------------------

final_control_row = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("source_name") == SOURCE_NAME) &
        (F.col("is_active") == True)
    )
    .first()
)

FINAL_PAYMENTS_WATERMARK = final_control_row["last_watermark"]


# ------------------------------------------------------------
# Capture current Bronze / Silver state
# ------------------------------------------------------------

final_bronze_df = spark.table(CONFIG_SOURCE_TABLE)
final_silver_df = spark.table(CONFIG_TARGET_TABLE)

final_bronze_count = final_bronze_df.count()
final_silver_count = final_silver_df.count()


# ------------------------------------------------------------
# Simulate pipeline restart
# ------------------------------------------------------------

final_restart_df = (
    final_bronze_df
    .withColumn(
        "_restart_watermark_ts",
        F.to_timestamp(F.col(CONFIG_WATERMARK_COLUMN))
    )
    .filter(
        F.col("_restart_watermark_ts") >
        F.lit(FINAL_PAYMENTS_WATERMARK)
    )
)

final_incremental_count = final_restart_df.count()


# ------------------------------------------------------------
# Validate controlled Payment keys remain unique in Silver
# ------------------------------------------------------------

update_payment_count = (
    final_silver_df
    .filter(F.col("payment_id") == UPDATE_PAYMENT_ID)
    .count()
)

insert_payment_count = (
    final_silver_df
    .filter(F.col("payment_id") == INSERT_PAYMENT_ID)
    .count()
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert final_incremental_count == 0, (
    f"Restart selected {final_incremental_count} "
    "previously processed Payments."
)

assert final_silver_count == 1501, (
    f"Expected 1501 Silver Payments, "
    f"found {final_silver_count}."
)

assert update_payment_count == 1, (
    f"{UPDATE_PAYMENT_ID} appears "
    f"{update_payment_count} times in Silver."
)

assert insert_payment_count == 1, (
    f"{INSERT_PAYMENT_ID} appears "
    f"{insert_payment_count} times in Silver."
)


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("Final Payments restart test completed.")
print("------------------------------------------")
print(f"Bronze Payments rows : {final_bronze_count}")
print(f"Silver Payments rows : {final_silver_count}")
print(f"Stored watermark     : {FINAL_PAYMENTS_WATERMARK}")
print(f"Incremental records  : {final_incremental_count}")
print(f"UPDATE key copies    : {update_payment_count}")
print(f"INSERT key copies    : {insert_payment_count}")

print()
print("==============================================")
print(" PAYMENTS RESTART / IDEMPOTENCY TEST PASSED")
print("==============================================")
print("No previously processed Payments were selected.")
print("No duplicate INSERT occurred.")
print("No repeated UPDATE occurred.")
print("Silver state remained unchanged.")
print("Payments incremental pipeline is restart-safe.")

StatementMeta(, 531909d3-505b-413b-8504-427c9306bbce, 36, Finished, Available, Finished, False)

Final Payments restart test completed.
------------------------------------------
Bronze Payments rows : 1503
Silver Payments rows : 1501
Stored watermark     : 2027-10-10 00:00:00
Incremental records  : 0
UPDATE key copies    : 1
INSERT key copies    : 1

 PAYMENTS RESTART / IDEMPOTENCY TEST PASSED
No previously processed Payments were selected.
No duplicate INSERT occurred.
No repeated UPDATE occurred.
Silver state remained unchanged.
Payments incremental pipeline is restart-safe.
